Understanding Config Box

In [2]:
dict1 = {"Name": "P1",
            "Age": 30}

In [3]:
dict1["Name"]

'P1'

In [6]:
# But we cannot write like this
# dict1.Name, it will give error because dictionary does not support attribute access.
# We have to use key access method to get the value from the dictionary.
# So to resolve this error, We can use ConfigBox to access the values from the dictionary using dot notation. 
# ConfigBox is a class that allows us to access the values from the dictionary using dot notation. 
# We can use it like this:
from box import ConfigBox

dict12 = ConfigBox({"Name": "P1",
                    "Age": 30})
dict12.Name

'P1'

Understanding the decorator @ensure_annotations

In [7]:
# @ensure_annotation is a decorator that is used to ensure that the function is called with the correct arguments.
#
# Lets make a function:
def add(a: int, b: int) -> int:
    return a + b

# Now if we call the function with the correct arguments, it will work fine.
add(1, 2)

3

In [9]:
# But if we call the function with the wrong arguments, it will give error.
add(1, "2")
# it produces error

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [10]:
# To resolve this error, we can use @ensure_annotation decorator to ensure that the function is called with the correct arguments.
from ensure import ensure_annotations
@ensure_annotations
def add(a: int, b: int) -> int:
    return a + b
add(1, 2)

3

In [11]:
add(1, "2")

EnsureError: Argument b of type <class 'str'> to <function add at 0x0000022E40DB3250> does not match annotation type <class 'int'>

In [12]:
pwd()

'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\ImageClass_MLOps_DVCpipeline\\deeplearning_MLOps_DVC\\research'

In [18]:
%pwd

'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\ImageClass_MLOps_DVCpipeline\\deeplearning_MLOps_DVC\\research'

In [19]:
# Change directory to main
import os
os.chdir("../")


In [20]:
pwd()

'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\ImageClass_MLOps_DVCpipeline\\deeplearning_MLOps_DVC'

In [24]:
# To create custom return type, we can create our own class and use it as return type in the function.
# This is to be done inside the entity module.

# First create a dataclass (which is a constant class) for defining configuration of data ingestion.

from dataclasses import dataclass
from pathlib import Path
from src.CNNclassifier import *

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL : str
    local_data_file : Path
    unzip_dir : Path


In [22]:
from src. CNNclassifier.utils.common import read_yaml, create_directories
from src.CNNclassifier.constants import *


In [ ]:
# This funciton will define the configurations for data ingestion before downloading and extracting data
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.dataingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )
        return data_ingestion_config
    


In [ ]:
# Here we will define class and its functions to download and extract data from the source URL
# This will be done in the data_ingestion module.

from urllib.request import request
import zipfile
from src.CNNclassifier.utils.common import get_size


class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config

    # Function to download the data from the source URL and save it to the local data file path
    def download_file(self):
        if os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(url = self.config.source_URL, filename =self.config.local_data_file)
            logger.info(f"{filename} downloaded with following info :\n{headers}")
        else:
            logger.info(f"{filename} already exists of size : {get_size(Path(self.config.local_data_file))}")

    # Function to extract data from the zipped folder stored by download_file()
    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        with zipfile.Zipfile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [ ]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e